## Hands-on Lab 1 : API Documentation Assistant

## Install Required Libraries

In [ ]:
%pip install langchain-ollama langgraph streamlit chromadb langchain-community langchain_chroma langchain_core langchain_classic unstructured pypdf

## Import Required Libraries

In [78]:
import os, json, shutil, uuid
from pathlib import Path
from datetime import datetime
import numpy as np
import socket

from langchain_ollama import OllamaLLM, OllamaEmbeddings, ChatOllama
from langchain_groq import ChatGroq
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_community.document_loaders import TextLoader, PyPDFLoader, UnstructuredMarkdownLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage
from langchain.agents import create_agent
from langgraph.checkpoint.memory import MemorySaver

CONFIG = {
    "docs_path": "api_docs",
    "db_path": "chroma_fixed_store",
    "llm_model": "llama3.1",
    "embedding_model": "nomic-embed-text",
}

os.makedirs(CONFIG["docs_path"], exist_ok=True)

# Function to detect if Ollama is running locally
def check_ollama():
    try:
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            s.settimeout(1)
            s.connect(("localhost", 11434))
            return True
    except:
        return False

OLLAMA_AVAILABLE = check_ollama()
if OLLAMA_AVAILABLE:
    print("✅ Local Ollama detected (running on port 11434)")
else:
    print("⚠️ Local Ollama NOT detected. Will use Cloud APIs (Groq + OpenAI) as fallback.")
    print("Make sure you set GROQ_API_KEY and  in environment variables.")

print("✅ Imports loaded")

✅ Imports loaded


### 1. Load Documents

In [79]:

def load_docs(path=CONFIG["docs_path"]):
    docs = []
    p = Path(path)

    # Load PDF files using PyPDFLoader
    for f in p.glob("*.pdf"):
        loaded = PyPDFLoader(str(f)).load()
        for d in loaded:
            d.metadata.update({
                "source_file": f.name,
                "full_path": str(f)
            })
        docs.extend(loaded)

    # Load text files using TextLoader
    for f in p.glob("*.txt"):
        loaded = TextLoader(str(f)).load()
        for d in loaded:
            d.metadata.update({
                "source_file": f.name,
                "full_path": str(f)
            })
        docs.extend(loaded)
    for f in p.glob("*.md"):

        try:
            # Best option: preserves markdown structure
            loaded = UnstructuredMarkdownLoader(str(f)).load()
        except:
            # Fallback: treat Markdown as plain text
            loaded = TextLoader(str(f)).load()

        for d in loaded:
            d.metadata.update({
                "source_file": f.name,
                "full_path": str(f)
            })

        docs.extend(loaded)

    print(f"✅ Loaded {len(docs)} documents")
    return docs

docs = load_docs()

✅ Loaded 3 documents


### 2. Chunking

In [80]:
splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=150)
splits = splitter.split_documents(docs)

for i, s in enumerate(splits):
    s.metadata["chunk_id"] = i

print("✅ Created", len(splits), "chunks")

✅ Created 7 chunks


### 3. Embeddings

In [81]:
if OLLAMA_AVAILABLE:
    embeddings = OllamaEmbeddings(model=CONFIG["embedding_model"])
    print("Embeddings loaded from local Ollama:", CONFIG["embedding_model"])
else:
    # Fallback to OpenAI embeddings
    # Set keys if not set
    if not os.getenv("OPENAI_API_KEY"):
        os.environ["OPENAI_API_KEY"] = "your-openai-api-key-here"
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
    print("Embeddings loaded from OpenAI (Cloud)")

Embeddings loaded: nomic-embed-text


### 4. VectorStore (Chroma)

In [82]:
#if os.path.exists(CONFIG["db_path"]):
#    shutil.rmtree(CONFIG["db_path"])


vectorstore = Chroma.from_documents(
    splits,
    embedding=embeddings,
    persist_directory=CONFIG["db_path"]
)

try:
    vectorstore._client.persist()
except:
    pass  # OK – Chroma persists automatically

retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

print("✅ Chroma vectorstore ready")


✅ Chroma vectorstore ready


### 5. Test Retrieval

In [83]:
def format_context(docs):
    lines = []
    for d in docs:
        text = d.page_content[:250].replace("\n", " ")   # f-string newline bug
        lines.append(f"[chunk {d.metadata['chunk_id']}] {d.metadata['source_file']}: {text}")
    return "\n".join(lines)

def retrieve(q: str):
    return retriever.invoke(q)  # LCEL-compatible retrieval


print("\n🔍 Retrieval Tests:")
for q in ["authenticate", "rate limit", "create user"]:
    docs = retrieve(q)
    print(f"\n{q} → {len(docs)} docs")
    print(format_context(docs)[:300] + "...")


🔍 Retrieval Tests:

authenticate → 5 docs
[chunk 4] authentication_guide.md: ### Key Security  - Keys expire after 90 days - Rotate keys regularly - Never commit keys to version control - Use environment variables to store keys - If compromised, regenerate immediately  ### Webhook Signing  Webhooks are signed with HMAC-SHA256
[chunk 1] auth...

rate limit → 5 docs
[chunk 0] api_guide.md: # API Documentation  ## Authentication  Authentication requires an API key. To obtain your API key: 1. Log into the dashboard 2. Navigate to Settings > API Keys 3. Click 'Generate New Key' 4. Copy the key immediately (it won't be shown again)  Import
[chunk 4] api_guide.md: #...

create user → 5 docs
[chunk 6] endpoints_reference.md: ### DELETE /api/v2/users/{id} Delete a user and all associated data.  Note: This is permanent and cannot be undone.  ## Organizations Endpoints  ### GET /api/v2/organizations List all organizations.  ### POST /api/v2/organizations Create organization
[chunk 3] endpo...


### 6. RAG Chain (LCEL)

In [84]:
if OLLAMA_AVAILABLE:
    llm = ChatOllama(model="llama3.1", temperature=0.1)
    print("LLM loaded from local Ollama: llama3.1")
else:
    # Fallback to Groq LLM
    if not os.getenv("GROQ_API_KEY"):
        os.environ["GROQ_API_KEY"] = "your-groq-api-key-here"
    llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.1)
    print("LLM loaded from Groq (Cloud): llama-3.1-8b-instant")

rag_prompt = ChatPromptTemplate.from_messages([
    ("system", """
Use ONLY the provided context. Give a detailed answer to the query
related to document and APIs.
If answer not found say: "I don't have that information in the documentation."
Cite chunks: Sources: [chunk X]
"""),
    ("human", "Context:\n{context}\n\nQuestion: {question}")
])

rag_chain = (
    {
        "context": retriever | format_context,
        "question": RunnablePassthrough()
    }
    | rag_prompt
    | llm
    | StrOutputParser()
)

print("✅ RAG chain ready")

✅ RAG chain ready


In [85]:
# Test RAG
for q in [
    "what is the document about?"
]:
    print("\nQ:", q)
    print("A:", rag_chain.invoke(q))


Q: what is the document about?
A: The document appears to be a reference guide for an API (Application Programming Interface). It outlines various endpoints, or URLs, that can be used to interact with the API. Specifically, it covers endpoints related to users and organizations.

There are multiple sections:

1. Endpoints related to deleting a user and all associated data.
2. Endpoints related to managing organizations, including listing all organizations and creating new ones.

Additionally, there is a separate section on pagination, which explains how to handle large responses by using cursor-based pagination.

Overall, the document provides a reference for developers who want to use the API to perform various operations.


### 7. Tools

In [89]:
@tool
def calculator(expr: str) -> str:
    """Safely evaluate a basic arithmetic expression."""
    allowed = set("0123456789+-*/(). eE")
    if any(c not in allowed for c in expr):
        return "Invalid characters"

    try:
        return str(eval(expr, {"__builtins__": {}}, {}))
    except Exception:
        return "Error"

@tool
def doc_search(query: str) -> str:
    """Search API documentation and return top matching chunks."""
    docs = retrieve(query)
    return format_context(docs)

tools = [calculator, doc_search]
print("Tools loaded:", [t.name for t in tools])

checkpointer = MemorySaver()

Tools loaded: ['calculator', 'doc_search']


### 8. CREATE AGENT

In [90]:
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="""
You are an API documentation assistant.

TOOLS AVAILABLE:
- calculator(expr): do arithmetic
- doc_search(query): retrieve API documentation sections

Rules:
- Use doc_search for all questions about API behavior.
- Use calculator only for math.
- Provide a final concise answer.
""",
)

agent_executor = agent

print("✅ v1 Agent ready — no AgentExecutor wrapper needed.")

✅ v1 Agent ready — no AgentExecutor wrapper needed.


In [91]:
# Usuage Example
config = {"configurable": {"thread_id": "agent-thread-1"}}

# SINGLE TURN
query = "Find rate limit from docs and calculate remaining calls (used=234 of free-1000)."
resp = agent_executor.invoke( {"messages": [{"role": "user", "content": query}]},config)
print("\nSingle-turn answer:\n", resp["messages"][-1].content)

# MULTI TURN (memory persists with same thread_id)
print("\n--- Multi-turn (memory) test ---")

resp1 = agent_executor.invoke(
    {"messages": [{"role": "user", "content": "How do I authenticate with the API?"}]},
    config
)
print("Assistant 1:", resp1["messages"][-1].content)

resp2 = agent_executor.invoke(
    {"messages": [{"role": "user", "content": "And what are the rate limits?"}]},
    config
)
print("\nAssistant 2:", resp2["messages"][-1].content)



Single-turn answer:
 To answer the question, we need to use doc_search to find information about rate limits in the API documentation, and then use calculator to calculate the remaining number of allowed calls.

First, let's search for the rate limit information:

{"name": "doc_search", "parameters": {"query": "rate limit"}} 

This will give us a response with relevant sections from the API documentation. Let's assume one of the responses is: "The rate limit is 1000 requests per minute."

Next, we can use calculator to calculate the remaining number of allowed calls:

{"name": "calculator", "parameters": {"expr": "1000 - 234"}} 

This will give us the result of the calculation.

--- Multi-turn (memory) test ---
Assistant 1: To authenticate with the API, you need to obtain an API key. Here's how:

1. Sign into your dashboard.
2. Go to Account Settings.
3. Select "API Keys" from the menu.

Your API key is scoped to specific permissions. Make sure to rotate your keys regularly and follow

### 9. Memory Test

In [92]:
session = {"configurable": {"thread_id": "session-A"}}

turn1 = agent_executor.invoke({"messages": [HumanMessage(content="Explain authentication")]}, session)
print("\nTurn1:", turn1["messages"][-1].content[:120], "...")

turn2 = agent_executor.invoke({"messages": [HumanMessage(content="And explain rate limits now")]}, session)
print("\nTurn2:", turn2["messages"][-1].content[:120], "...")


Turn1: API key authentication is the primary method of authentication. To obtain an API key:

1. Sign into your dashboard
2. Go ...

Turn2: The API has a rate limit in place to prevent overwhelming responses. When the rate limit is reached, the API returns a 4 ...


### 10. Guardrails

In [93]:
BLOCKED = {"password", "secret", "private key", "admin"}

def safe_ask(query):
    q = query.lower()
    if any(b in q for b in BLOCKED):
        return "🚫 Blocked query"
    if len(q.split()) < 3:
        return "❌ Too short"
    return rag_chain.invoke(query)

print("\nGuardrail Test:")
for q in ["show admin password", "how authenticate api"]:
    print(q, "→", safe_ask(q))


Guardrail Test:
show admin password → 🚫 Blocked query
how authenticate api → According to [chunk 3] and [chunk 0], the primary authentication method is API Key Authentication. To authenticate an API, you need to use your API key, which can be obtained by following these steps:

1. Sign into the dashboard
2. Go to Account Settings
3. Select "API Keys" from the menu

Once you have your API key, you can use it to authenticate your API requests.

Additionally, [chunk 4] provides some best practices for securing your API keys, such as rotating them regularly and never committing them to version control.


### 11. Retrieval Evaluation

In [ ]:
ground_truth = {
    "How do I authenticate?": ["api-docs-handbook.pdf"],
    "What is rate limit?": ["api-docs-handbook.pdf"],
}

def eval_retrieval(query):
    docs = retrieve(query)
    retrieved = [d.metadata["source_file"] for d in docs]
    relevant = ground_truth.get(query, [])

    recall = len(set(retrieved) & set(relevant)) / len(relevant)
    precision = len(set(retrieved) & set(relevant)) / max(1, len(retrieved))

    return {
        "recall": recall,
        "precision": precision,
        "retrieved": retrieved[:3]}
        

print("\n📊 Retrieval Metrics:")
for q in ground_truth:
    print(q, eval_retrieval(q))


📊 Retrieval Metrics:
How do I authenticate? {'recall': 0.0, 'precision': 0.0, 'retrieved': ['authentication_guide.md', 'authentication_guide.md', 'api_guide.md']}
What is rate limit? {'recall': 0.0, 'precision': 0.0, 'retrieved': ['api_guide.md', 'api_guide.md', 'endpoints_reference.md']}
